<div dir="rtl" style="white-space: normal; line-height: 1.8; text-align: right;">

<h1>ارزیابی ارزش RAG قرآن</h1>

<p>
هدف این Notebook بررسی این است که آیا بازیابی معنایی و انتخاب منبع با مدل زبانی،
نسبت به جست‌وجوی ساده کلمه‌ای ارزش واقعی ایجاد می‌کند یا نه.
</p>

<p>سه روش مقایسه می‌شوند:</p>

<ol>
  <li>جست‌وجوی کلمه‌ای ساده</li>
  <li>بازیابی معنایی با Embedding</li>
  <li>بازیابی معنایی همراه با انتخاب منبع توسط Qwen</li>
</ol>

<p>
ارزیابی روی سؤال‌های مستقیم، غیرمستقیم و مفهومی انجام می‌شود.
</p>

</div>

In [1]:
# Cell 1 - Load evaluation resources

from pathlib import Path
import json
import re

import numpy as np
from sentence_transformers import SentenceTransformer


project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

dataset_file = (
    project_root
    / "data"
    / "processed"
    / "quran_dataset_clean.json"
)

embeddings_file = (
    project_root
    / "data"
    / "embeddings"
    / "quran_embeddings.npy"
)

with open(dataset_file, "r", encoding="utf-8") as file:
    quran_data = json.load(file)

quran_embeddings = np.load(embeddings_file)

embedding_model_name = "intfloat/multilingual-e5-small"

embedding_model = SentenceTransformer(
    embedding_model_name,
    device="cpu"
)

print("Verses:", len(quran_data))
print("Embeddings shape:", quran_embeddings.shape)
print("Evaluation resources are ready.")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Verses: 6236
Embeddings shape: (6236, 384)
Evaluation resources are ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
Cell 2 — Keyword Search را اضافه کن.

این بخش یک جست‌وجوی کلمه‌ای ساده می‌سازد تا بعداً با بازیابی معنایی مقایسه‌اش کنیم.
</div>

In [2]:
# Cell 2 - Build a simple keyword search baseline

def normalize_persian_text(text):
    """
    Normalize common Persian and Arabic character variants.
    """
    normalized = text.strip().lower()

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
    }

    for old_char, new_char in replacements.items():
        normalized = normalized.replace(old_char, new_char)

    normalized = re.sub(r"[^\w\s]", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized)

    return normalized.strip()


def keyword_search(query, top_k=5):
    """
    Rank verses using simple keyword overlap.
    """
    normalized_query = normalize_persian_text(query)
    query_terms = set(normalized_query.split())

    results = []

    for item in quran_data:
        searchable_text = " ".join([
            item["fooladvand_clean"],
            item["ansarian_clean"],
        ])

        normalized_text = normalize_persian_text(searchable_text)
        text_terms = set(normalized_text.split())

        matched_terms = query_terms.intersection(text_terms)
        score = len(matched_terms)

        if score > 0:
            results.append({
                "score": score,
                "matched_terms": sorted(matched_terms),
                "surah": item["surah"],
                "ayah": item["ayah"],
                "arabic": item["arabic"],
                "fooladvand": item["fooladvand"],
                "ansarian": item["ansarian"],
            })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:top_k]


print("Keyword search baseline is ready.")

Keyword search baseline is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
Cell 3 — Semantic Retrieval را اضافه کن.

در این بخش، همان سؤال‌ها با embedding جست‌وجو می‌شوند تا ببینیم بازیابی معنایی نسبت به تطبیق کلمه‌ای چه تفاوتی دارد.
</div>

In [3]:
# Cell 3 - Build semantic retrieval

def semantic_search(query, top_k=5):
    """
    Rank verses using semantic similarity.
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    query_embedding = embedding_model.encode(
        [f"query: {query.strip()}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )[0]

    scores = quran_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = quran_data[index]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "surah": item["surah"],
            "ayah": item["ayah"],
            "arabic": item["arabic"],
            "fooladvand": item["fooladvand"],
            "ansarian": item["ansarian"],
        })

    return results


print("Semantic retrieval is ready.")

Semantic retrieval is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
حالا Cell 4 — Define Evaluation Questions را اضافه کن.

این بخش مجموعه‌ای از سؤال‌های مستقیم، غیرمستقیم و مفهومی می‌سازد تا هر دو روش جست‌وجوی کلمه‌ای و معنایی روی یک معیار مشترک سنجیده شوند.
</div>

In [4]:
# Cell 4 - Define evaluation questions

evaluation_cases = [
    {
        "id": "Q1",
        "type": "direct",
        "question": "خدا توبه‌کنندگان را دوست دارد",
        "expected_refs": [(2, 222)],
    },
    {
        "id": "Q2",
        "type": "indirect",
        "question": "چه کسانی محبوب خدا هستند؟",
        "expected_refs": [(2, 222), (3, 134), (3, 146), (5, 13), (61, 4)],
    },
    {
        "id": "Q3",
        "type": "conceptual",
        "question": "قرآن درباره صبر در سختی چه می‌گوید؟",
        "expected_refs": [],
    },
    {
        "id": "Q4",
        "type": "indirect",
        "question": "آیا خدا کسانی را که در راه او می‌جنگند دوست دارد؟",
        "expected_refs": [(61, 4)],
    },
    {
        "id": "Q5",
        "type": "conceptual",
        "question": "در قرآن چه کسانی بخشیده می‌شوند؟",
        "expected_refs": [],
    },
]

print("Evaluation cases:", len(evaluation_cases))

for case in evaluation_cases:
    print(
        case["id"],
        "-",
        case["type"],
        "-",
        case["question"]
    )

Evaluation cases: 5
Q1 - direct - خدا توبه‌کنندگان را دوست دارد
Q2 - indirect - چه کسانی محبوب خدا هستند؟
Q3 - conceptual - قرآن درباره صبر در سختی چه می‌گوید؟
Q4 - indirect - آیا خدا کسانی را که در راه او می‌جنگند دوست دارد؟
Q5 - conceptual - در قرآن چه کسانی بخشیده می‌شوند؟


<div style="direction: rtl; white-space: normal; line-height: 1;">
حالا Cell 5 — Compare Keyword and Semantic Results را اضافه کن.

این سلول هر سؤال را با هر دو روش اجرا می‌کند و فقط شماره سوره، آیه و امتیاز را کنار هم نشان می‌دهد تا تفاوت اولیه را ببینیم.
</div>

In [5]:
# Cell 5 - Compare keyword and semantic retrieval

def verse_reference(result):
    """
    Return a verse reference as a tuple.
    """
    return result["surah"], result["ayah"]


comparison_results = []

for case in evaluation_cases:
    question = case["question"]

    keyword_results = keyword_search(
        question,
        top_k=5
    )

    semantic_results = semantic_search(
        question,
        top_k=5
    )

    comparison_results.append({
        "id": case["id"],
        "type": case["type"],
        "question": question,
        "expected_refs": case["expected_refs"],
        "keyword_results": keyword_results,
        "semantic_results": semantic_results,
    })

    print("=" * 80)
    print(
        case["id"],
        "-",
        case["type"],
        "-",
        question
    )

    print("\nKeyword Search:")

    if keyword_results:
        for rank, result in enumerate(
            keyword_results,
            start=1
        ):
            print(
                f"{rank}. "
                f"{result['surah']}:{result['ayah']} "
                f"| score={result['score']} "
                f"| matched={result['matched_terms']}"
            )
    else:
        print("No keyword results found.")

    print("\nSemantic Search:")

    for result in semantic_results:
        print(
            f"{result['rank']}. "
            f"{result['surah']}:{result['ayah']} "
            f"| score={result['score']:.4f}"
        )

print("\nInitial comparison is complete.")

Q1 - direct - خدا توبه‌کنندگان را دوست دارد

Keyword Search:
1. 2:222 | score=5 | matched=['توبه', 'خدا', 'دارد', 'دوست', 'را']
2. 3:159 | score=5 | matched=['خدا', 'دارد', 'دوست', 'را', 'کنندگان']
3. 7:31 | score=5 | matched=['خدا', 'دارد', 'دوست', 'را', 'کنندگان']
4. 28:76 | score=5 | matched=['خدا', 'دارد', 'دوست', 'را', 'کنندگان']
5. 49:12 | score=5 | matched=['توبه', 'خدا', 'دارد', 'دوست', 'را']

Semantic Search:
1. 40:7 | score=0.8804
2. 61:4 | score=0.8770
3. 2:222 | score=0.8761
4. 25:70 | score=0.8738
5. 9:112 | score=0.8718
Q2 - indirect - چه کسانی محبوب خدا هستند؟

Keyword Search:
1. 4:63 | score=4 | matched=['خدا', 'هستند', 'چه', 'کسانی']
2. 4:69 | score=4 | matched=['خدا', 'هستند', 'چه', 'کسانی']
3. 11:18 | score=4 | matched=['خدا', 'هستند', 'چه', 'کسانی']
4. 31:11 | score=4 | matched=['خدا', 'هستند', 'چه', 'کسانی']
5. 34:23 | score=4 | matched=['خدا', 'هستند', 'چه', 'کسانی']

Semantic Search:
1. 61:4 | score=0.8541
2. 22:59 | score=0.8538
3. 37:159 | score=0.8524
4. 42:19

<div style="direction: rtl; white-space: normal; line-height: 1;">
Keyword Search شدیداً به کلمات عمومی مثل «خدا»، «در»، «چه»، «می» وزن می‌دهد و نتایج نامرتبط بالا می‌آورد.
Semantic Search بهتر مفهوم را می‌فهمد، اما هنوز همیشه آیه درست را رتبه ۱ نمی‌آورد.
مثلاً در Q1 آیه صحیح 2:222 در رتبه ۳ است.
در Q4 آیه صحیح 61:4 در رتبه ۲ است.

پس توجیه RAG اینجا روشن می‌شود:
بازیابی معنایی کاندیداهای بهتر می‌دهد و Qwen باید از بین آن‌ها منبع درست را انتخاب کند.
</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">
انتخاب منبع با Qwen را به ارزیابی وصل کنیم.

این سلول تنظیمات Arvan را بارگذاری می‌کند و تابع انتخاب منبع را می‌سازد؛ هنوز پاسخ تولید نمی‌کنیم.
</div>

In [6]:
# Cell 6 - Build Qwen source selection for evaluation

import os
import json

from dotenv import load_dotenv
from openai import OpenAI


load_dotenv(project_root / ".env")

arvan_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_base_url = os.getenv("ARVAN_CHAT_URL")
arvan_model = os.getenv(
    "ARVAN_CHAT_MODEL",
    "Qwen3-30B-A3B"
)

if not arvan_api_key:
    raise ValueError("ARVAN_AI_API_KEY is missing.")

if not arvan_base_url:
    raise ValueError("ARVAN_CHAT_URL is missing.")

qwen_client = OpenAI(
    api_key=arvan_api_key,
    base_url=arvan_base_url
)


def select_sources_with_qwen(
    question,
    semantic_results
):
    """
    Select only the retrieved verses that directly answer the question.
    """
    candidates = []

    for result in semantic_results:
        candidates.append(
            {
                "rank": result["rank"],
                "surah": result["surah"],
                "ayah": result["ayah"],
                "fooladvand": result["fooladvand"],
                "ansarian": result["ansarian"],
            }
        )

    prompt = f"""
You are evaluating Quran retrieval results.

Question:
{question}

Candidate verses:
{json.dumps(candidates, ensure_ascii=False, indent=2)}

Select only verses that directly answer the question.

Rules:
- Use only the candidate verses.
- Do not generate an answer.
- Do not add any verse.
- Return valid JSON only.
- If none directly answer the question, return an empty list.

Required format:
{{"selected_ranks": [1, 2]}}
""".strip()

    response = qwen_client.chat.completions.create(
        model=arvan_model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=100
    )

    raw_output = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    try:
        parsed_output = json.loads(raw_output)
    except json.JSONDecodeError:
        return []

    selected_ranks = parsed_output.get(
        "selected_ranks",
        []
    )

    selected_results = [
        result
        for result in semantic_results
        if result["rank"] in selected_ranks
    ]

    return selected_results


print("Qwen source selection is ready.")

Qwen source selection is ready.


Run Qwen Selection Evaluation

<div style="direction: rtl; white-space: normal; line-height: 1;">
این سلول برای هر سؤال، نتایج Semantic Search را می‌گیرد و Qwen از بین همان ۵ آیه، منابع مستقیم را انتخاب می‌کند.
</div>

In [7]:
# Cell 7 - Evaluate Qwen source selection

for comparison in comparison_results:
    question = comparison["question"]
    semantic_results = comparison["semantic_results"]

    selected_results = select_sources_with_qwen(
        question,
        semantic_results
    )

    comparison["qwen_selected_results"] = selected_results

    print("=" * 80)
    print(
        comparison["id"],
        "-",
        comparison["type"],
        "-",
        question
    )

    print("\nSemantic candidates:")

    for result in semantic_results:
        print(
            f"{result['rank']}. "
            f"{result['surah']}:{result['ayah']} "
            f"| score={result['score']:.4f}"
        )

    print("\nQwen selected sources:")

    if selected_results:
        for result in selected_results:
            print(
                f"Rank {result['rank']} "
                f"| {result['surah']}:{result['ayah']}"
            )
    else:
        print("No source selected.")

print("\nQwen selection evaluation is complete.")

Q1 - direct - خدا توبه‌کنندگان را دوست دارد

Semantic candidates:
1. 40:7 | score=0.8804
2. 61:4 | score=0.8770
3. 2:222 | score=0.8761
4. 25:70 | score=0.8738
5. 9:112 | score=0.8718

Qwen selected sources:
Rank 3 | 2:222
Rank 4 | 25:70
Q2 - indirect - چه کسانی محبوب خدا هستند؟

Semantic candidates:
1. 61:4 | score=0.8541
2. 22:59 | score=0.8538
3. 37:159 | score=0.8524
4. 42:19 | score=0.8498
5. 16:128 | score=0.8495

Qwen selected sources:
Rank 1 | 61:4
Rank 5 | 16:128
Q3 - conceptual - قرآن درباره صبر در سختی چه می‌گوید؟

Semantic candidates:
1. 46:35 | score=0.8607
2. 46:8 | score=0.8556
3. 54:8 | score=0.8547
4. 41:41 | score=0.8540
5. 11:17 | score=0.8539

Qwen selected sources:
Rank 1 | 46:35
Q4 - indirect - آیا خدا کسانی را که در راه او می‌جنگند دوست دارد؟

Semantic candidates:
1. 4:84 | score=0.8826
2. 61:4 | score=0.8805
3. 4:90 | score=0.8761
4. 3:13 | score=0.8755
5. 3:140 | score=0.8741

Qwen selected sources:
Rank 2 | 61:4
Q5 - conceptual - در قرآن چه کسانی بخشیده می‌شون

<div style="direction: rtl; white-space: normal; line-height: 1;">
Q1 آیه درست 2:222 را انتخاب کرده، ولی 25:70 را هم اضافه کرده که پاسخ مستقیم سؤال نیست.
Q2 آیه 61:4 درست است، ولی 16:128 احتمالاً پاسخ مستقیم «محبوب خدا» نیست.
Q4 کاملاً درست عمل کرده و فقط 61:4 را انتخاب کرده.
Q3 و Q5 باید متن آیات را ببینیم تا دقیق قضاوت کنیم.

پس فعلاً نتیجه این نیست که RAG عالی است؛ نتیجه این است که:

بازیابی معنایی کاندیداهای بهتری می‌دهد، اما انتخاب نهایی با Qwen هنوز نیاز به ارزیابی و محدودسازی دارد.
</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">
Cell 8 — Inspect Selected Verse Texts را اضافه کن تا متن واقعی آیات انتخاب‌شده را ببینیم و دستی قضاوت کنیم.
</div>

In [8]:
# Cell 8 - Inspect Qwen selected verse texts

for comparison in comparison_results:
    selected_results = comparison.get(
        "qwen_selected_results",
        []
    )

    print("=" * 80)
    print(
        comparison["id"],
        "-",
        comparison["question"]
    )

    if not selected_results:
        print("\nNo source selected.")
        continue

    for result in selected_results:
        print(
            f"\nSurah {result['surah']}, "
            f"Ayah {result['ayah']}"
        )

        print("\nFooladvand:")
        print(result["fooladvand"])

        print("\nAnsarian:")
        print(result["ansarian"])

        print("\n" + "-" * 60)

print("\nSelected verse inspection is complete.")

Q1 - خدا توبه‌کنندگان را دوست دارد

Surah 2, Ayah 222

Fooladvand:
از تو در باره عادت ماهانه [زنان‌] می‌پرسند، بگو: «آن، رنجی است. پس هنگام عادت ماهانه، از [آمیزش با] زنان کناره گیری کنید، و به آنان نزدیک نشوید تا پاک شوند. پس چون پاک شدند، از همان جا که خدا به شما فرمان داده است، با آنان آمیزش کنید.» خداوند توبه‌کاران و پاکیزگان را دوست می‌دارد.

Ansarian:
از تو درباره حیض می پرسند، بگو: حیض، حالت ناملایم و زیان باری است؛ پس در زمان حیض از [آمیزش با] زنان کناره گیری کنید، و با آنان نزدیکی ننمایید تا پاک شوند؛ و هنگامی که پاک شدند از جایی که خدا به شما فرمان داده با آنان آمیزش کنید. یقیناً خدا کسانی را که بسیار توبه می کنند، و کسانی را که خود را [با پذیرش انواع پاکی ها از همه آلودگی ها] پاکیزه می کنند. دوست دارد.

------------------------------------------------------------

Surah 25, Ayah 70

Fooladvand:
مگر کسی که توبه کند و ایمان آورد و کار شایسته کند. پس خداوند بدیهایشان را به نیکیها تبدیل می‌کند، و خدا همواره آمرزنده مهربان است.

Ansarian:
مگر آنان که توبه کنند و ایمان آورند و کار

<div style="direction: rtl; white-space: normal; line-height: 1;">
و این‌ها با هم قاطی می‌شوند. خروجی Q5 دقیقاً نشان داد Qwen گاهی به خاطر شباهت واژه‌ای آیات اشتباه مثل 15:91 و 6:159 را انتخاب می‌کند.

پس این Notebook را همین‌جا با یک نتیجه موقت جمع می‌کنیم:

بازیابی معنایی نسبت به جست‌وجوی کلمه‌ای کاندیداهای مرتبط‌تری تولید می‌کند، اما کیفیت انتخاب نهایی به مدل زبانی وابسته است. Qwen برای فارسی و تشخیص دقیق ارتباط آیات محدودیت دارد؛ بنابراین نتایج انتخاب منبع، ارزیابی نهایی معماری RAG محسوب نمی‌شوند.
</div>

<div dir="rtl" style="white-space: normal; line-height: 1.8; text-align: right;">

<h2>نتیجه اولیه ارزیابی</h2>

<p>
نتایج این ارزیابی نشان داد که جست‌وجوی کلمه‌ای ساده، به دلیل وزن دادن به واژه‌های عمومی مانند
«خدا»، «در»، «چه» و «می»، در بسیاری از سؤال‌ها نتایج نامرتبط تولید می‌کند.
</p>

<p>
بازیابی معنایی در بیشتر موارد آیات مرتبط‌تری را در میان نتایج ابتدایی قرار داد، اما آیه صحیح
همیشه در رتبه اول نبود.
</p>

<p>
استفاده از Qwen برای انتخاب منبع، در بعضی سؤال‌ها مانند سؤال مربوط به سوره صف آیه ۴،
نتیجه دقیق‌تری ایجاد کرد؛ اما در برخی موارد نیز آیات غیرمستقیم یا نامرتبط را انتخاب کرد.
</p>

<p>
بنابراین نتیجه فعلی این است که معماری RAG نسبت به جست‌وجوی کلمه‌ای مزیت دارد،
اما کیفیت انتخاب نهایی به توانایی مدل زبانی در درک فارسی و تشخیص ارتباط دقیق آیات وابسته است.
</p>

<p>
از آنجا که در این نسخه از Qwen به دلیل هزینه کمتر استفاده شده است، این نتایج نباید به عنوان
ارزیابی نهایی معماری در نظر گرفته شوند. در آینده می‌توان فقط بخش انتخاب منبع را با یک مدل
قوی‌تر جایگزین و دوباره ارزیابی کرد، بدون نیاز به بازسازی بخش بازیابی و embedding.
</p>

<h3>جمع‌بندی</h3>

<ul>
  <li>جست‌وجوی کلمه‌ای برای پرسش‌های مفهومی کافی نیست.</li>
  <li>بازیابی معنایی کاندیداهای مناسب‌تری تولید می‌کند.</li>
  <li>مدل زبانی باید فقط نقش انتخاب منبع را داشته باشد.</li>
  <li>متن نهایی پاسخ باید مستقیماً از منابع معتبر قرآن نمایش داده شود.</li>
  <li>ارزیابی نهایی انتخاب منبع به مدل فارسی قوی‌تری موکول می‌شود.</li>
</ul>

</div>